# Calculating an XMY

### Step 0: Set-up

Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
from climakitae.util.utils import (
    convert_to_local_time,
    get_closest_gridcell,
)
from climakitae.core.data_export import write_tmy_file
from climakitae.core.data_interface import get_data
#from climakitaegui.explore.typical_meteorological_year import plot_one_var_cdf

import pandas as pd
import xarray as xr
import numpy as np
import pkg_resources
from tqdm.auto import tqdm  # Progress bar

import warnings
warnings.filterwarnings("ignore")

### Step 1: Grab and process all required input data

The [TMY3 method](https://www.nrel.gov/docs/fy08osti/43156.pdf) selects a "typical" month based on ten daily variables: max, min, and mean air and dew point temperatures, max and mean wind speed, global irradiance and direct irradiance.  

#### Step 1a: Select location of interest
TMYs are calculated for a specific location of interest, like a building or power plant. Here, we will use a known weather station location, via their latitude and longitude to extract the data that we need to calculate the TMY. In the example below, we will look specifically at Los Angeles International Airport, but will note in the code below how you can provide your own location coordinates too. 

In [2]:
# read in station file of CA HadISD stations
stn_file = pkg_resources.resource_filename("climakitae", "data/hadisd_stations.csv")
stn_file = pd.read_csv(stn_file, index_col=[0])
stn_file.head(5)

,state,station,city,ID,LAT_Y,LON_X,station id,elevation
0,CA,Bakersfield Meadows Field (KBFL),Bakersfield,KBFL,35.43424,-119.05524,72384023155,149.3
1,CA,Blythe Asos (KBLH),Blythe,KBLH,33.61876,-114.71451,74718823158,120.4
2,CA,Burbank-Glendale-Pasadena Airport (KBUR),Burbank,KBUR,34.19966,-118.36543,72288023152,222.7
3,CA,Needles Airport (KEED),Needles,KEED,34.76783,-114.61842,72380523179,270.6
4,CA,Fresno Yosemite International Airport (KFAT),Fresno,KFAT,36.77999,-119.72016,72389093193,101.9


In [3]:
# grab airport
stn_name = "Bakersfield Meadows Field (KBFL)"
stn_code = stn_file.loc[stn_file["station"] == stn_name]["station id"].item()
one_station = stn_file.loc[stn_file["station"] == stn_name]
stn_lat = one_station.LAT_Y.item()
stn_lon = one_station.LON_X.item()
stn_state = one_station.state.item()
stn_lat, stn_lon

(35.43424, -119.05524)

In [4]:
print(stn_code)

72384023155


Alternatively, you may want to provide your own location instead of one of the HadISD stations above. If so, uncomment the cell below by removing the `#` symbol and modifying the lines of code. Note, with custom locations, if an elevation is not provided, a default value of 0.0 m will be used. 

In [5]:
## provide your own location, via latitude and longitude coordinates
# stn_lat = YOUR_LAT_HERE
# stn_lon = YOUR_LON_HERE
# stn_state = 'YOUR_STATE_HERE'
# stn_name = 'YOUR_STATION_NAME_HERE'
# stn_code = 'custom'
# stn_elev = YOUR_ELEV_HERE # meters

#### Step 1b: Select time frame of interest
The second required input for generating a TMY dataset is the **time frame of interest**. The recommended minimum number of input years for a TMY dataset is 15-20 years worth of daily data; we will use 30 years to represent a standard climatological period. For data post-2014, we will utilize SSP 3-7.0, although scenario selection in the near-future is relatively independent. If calculating a TMY for the far-future, **carefully consider which scenario SSP to include**, as there will be **significant** differences present. 

We will also process the data for our designated station location (latitude, and longitude) at 3 km over the <span style="color:#FF0000">1990-2020 period</span> as an example. **Note**: An additional year (2021) is also loaded to pad the end of the dataset after converting to local time in the next few steps -- this is done for you when subsetting for the data. 

In [6]:
# selected reference period
start_year = 1990
end_year = 2020

#### Step 1c: Retrieve variables in catalog
It is important to note that not all models in the Cal-Adapt: Analytics Engine have the solar variables critical for TMY file generation - in fact, only 4 do! We will carefully subset our variables to ensure that the same 4 models are selected for consistency. 

Lastly, because the dynamically downscaled WRF data in the Cal-Adapt: Analytics Engine is in UTC time, we'll convert to the timezone of the station we've selected. This is particularly important for the timing of solar radiation max and min, which is a critical piece of a TMY. The handy `convert_to_local_time` function corrects for this, and ensures that all data are within the same daily timestamp.

In [7]:
# selected models
data_models = [
    "WRF_EC-Earth3_r1i1p1f1",
    "WRF_MPI-ESM1-2-HR_r3i1p1f1",
    # "WRF_TaiESM1_r1i1p1f1",
    # "WRF_MIROC6_r1i1p1f1",
]

Now that we have set up the model list, let's start retrieving data! We will need to calculate summary statistics and reduce to daily timescales for the following variables:

In [8]:
# air temperature
temp_data = get_data(
    variable="Air Temperature at 2m",
    resolution="3 km",
    timescale="hourly",
    data_type="Gridded",
    units="degC",
    latitude=(stn_lat - 0.05, stn_lat + 0.05),
    longitude=(stn_lon - 0.06, stn_lon + 0.06),
    area_average="Yes",
    scenario=["Historical Climate", "SSP 3-7.0"],
    time_slice=(start_year, end_year + 1),
)

temp_data = convert_to_local_time(
    temp_data, stn_lon, stn_lat
)  # convert to local timezone, provide lon/lat because area average data lacks coordinates
temp_data = temp_data.sel({"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")})
temp_data = temp_data.sel(simulation=data_models)

# max air temp
max_airtemp_data = temp_data.resample(time="1D").max()  # daily max air temp
max_airtemp_data.name = "Daily max air temperature"  # rename for clarity

# min air temp
min_airtemp_data = temp_data.resample(time="1D").min()  # daily min air temp
min_airtemp_data.name = "Daily min air temperature"  # rename for clarity

# mean air temp
mean_airtemp_data = temp_data.resample(time="1D").mean()  # daily mean air temp
mean_airtemp_data.name = "Daily mean air temperature"  # rename for clarity
mean_airtemp_data = mean_airtemp_data.compute()

Data converted to America/Los_Angeles timezone.


In [16]:
temp_data.time

<xarray.DataArray 'time' (time: 271752)> Size: 2MB
array(['1990-01-01T00:00:00.000000000', '1990-01-01T01:00:00.000000000',
       '1990-01-01T02:00:00.000000000', ..., '2020-12-31T21:00:00.000000000',
       '2020-12-31T22:00:00.000000000', '2020-12-31T23:00:00.000000000'],
      dtype='datetime64[ns]')
Coordinates:
    Lambert_Conformal  int64 8B 0
  * time               (time) datetime64[ns] 2MB 1990-01-01 ... 2020-12-31T23...

Retrieve and calculate max and mean wind speed:

In [9]:
# wind speed
ws_data = get_data(
    variable="Wind speed at 10m",
    resolution="3 km",
    timescale="hourly",
    data_type="Gridded",
    units="m s-1",
    latitude=(stn_lat - 0.05, stn_lat + 0.05),
    longitude=(stn_lon - 0.06, stn_lon + 0.06),
    area_average="Yes",
    scenario=["Historical Climate", "SSP 3-7.0"],
    time_slice=(start_year, end_year + 1),
)

ws_data = convert_to_local_time(
    ws_data, stn_lon, stn_lat
)  # convert to local timezone, provide lon/lat because area average data lacks coordinates
ws_data = ws_data.sel(
    {"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")}
)  # your desired time slice in local time
ws_data = ws_data.sel(simulation=data_models)

# max wind speed
max_windspd_data = ws_data.resample(time="1D").max()  # daily max wind speed
max_windspd_data.name = "Daily max wind speed"  # rename for clarity

# mean wind speed
mean_windspd_data = ws_data.resample(time="1D").mean()  # daily mean wind speed
mean_windspd_data.name = "Daily mean wind speed"  # rename for clarity

Data converted to America/Los_Angeles timezone.


In [12]:
mean_windspd_data

<xarray.DataArray 'Daily mean wind speed' (scenario: 1, simulation: 2,
                                           time: 11323, y: 4, x: 5)> Size: 2MB
dask.array<transpose, shape=(1, 2, 11323, 4, 5), dtype=float32, chunksize=(1, 1, 1, 4, 5), chunktype=numpy.ndarray>
Coordinates:
  * simulation         (simulation) <U26 208B 'WRF_EC-Earth3_r1i1p1f1' 'WRF_M...
  * scenario           (scenario) <U22 88B 'Historical + SSP 3-7.0'
  * x                  (x) float64 40B -4.095e+06 -4.092e+06 ... -4.083e+06
  * y                  (y) float64 32B 1.01e+06 1.013e+06 1.016e+06 1.019e+06
    Lambert_Conformal  int64 8B 0
    lat                (y, x) float32 80B dask.array<chunksize=(4, 5), meta=np.ndarray>
    lon                (y, x) float32 80B dask.array<chunksize=(4, 5), meta=np.ndarray>
  * time               (time) datetime64[ns] 91kB 1990-01-01 ... 2020-12-31
Attributes:
    variable_id:           wind_speed_derived
    extended_description:  Wind speed at 10 meters above the Earth's surface....
    units:                 m s-1
    data_type:             Gridded
    resolution:            3 km
    frequency:             hourly
    location_subset:       ['coordinate selection']
    approach:              Time
    downscaling_method:    Dynamical
    institution:           Multiple
    grid_mapping:          Lambert_Conformal
    timezone:              America/Los_Angeles

Retrieve and calculate max, min, and mean dew point temperature:

In [10]:
# dew point temperature
dewpt_data = get_data(
    variable="Dew point temperature",
    resolution="3 km",
    timescale="hourly",
    data_type="Gridded",
    units="degC",
    latitude=(stn_lat - 0.05, stn_lat + 0.05),
    longitude=(stn_lon - 0.06, stn_lon + 0.06),
    area_average="Yes",
    scenario=["Historical Climate", "SSP 3-7.0"],
    time_slice=(start_year, end_year + 1),
)

dewpt_data = convert_to_local_time(
    dewpt_data, stn_lon, stn_lat
)  # convert to local timezone, provide lon/lat because area average data lacks coordinates
dewpt_data = dewpt_data.sel({"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")})
dewpt_data = dewpt_data.sel(simulation=data_models)

# max dew point
max_dewpt_data = dewpt_data.resample(time="1D").max()  # daily max dewpoint temp
max_dewpt_data.name = "Daily max dewpoint temperature"  # rename for clarity

# min dew point
min_dewpt_data = dewpt_data.resample(time="1D").min()  # daily min dewpoint temp
min_dewpt_data.name = "Daily min dewpoint temperature"  # rename for clarity

# mean dew point
mean_dewpt_data = dewpt_data.resample(time="1D").mean()  # daily mean dewpoint temp
mean_dewpt_data.name = "Daily mean dewpoint temperature"  # rename for clarity

Data converted to America/Los_Angeles timezone.


Next, retrieve global horizontal irradiance. GHI is within the Analytics Engine catalog at daily resolutions, but for the TMY methodology, we need to calculate the total accumulated GHI received over the course of the day, so we will retrieve hourly data instead and calculate the necessary information below. The same process is repeated for the direct normal irradiance. 

In [11]:
# global irradiance
global_irradiance_data = get_data(
    variable="Instantaneous downwelling shortwave flux at bottom",
    resolution="3 km",
    timescale="hourly",
    data_type="Gridded",
    latitude=(stn_lat - 0.05, stn_lat + 0.05),
    longitude=(stn_lon - 0.06, stn_lon + 0.06),
    area_average="Yes",
    scenario=["Historical Climate", "SSP 3-7.0"],
    time_slice=(start_year, end_year + 1),
)

global_irradiance_data = convert_to_local_time(
    global_irradiance_data, stn_lon, stn_lat
)  # convert to local timezone, provide lon/lat because area average data lacks coordinates
global_irradiance_data = global_irradiance_data.sel(
    {"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")}
)
global_irradiance_data = global_irradiance_data.sel(simulation=data_models)

global_irradiance_data.name = "Global horizontal irradiance"  # rename for clarity
# total global horizontal irradiance (accumulation of hourly data over a 24-hour period)
total_ghi_data = global_irradiance_data.resample(time="1D").sum()

Data converted to America/Los_Angeles timezone.


In [12]:
# direct normal irradiance
direct_irradiance_data = get_data(
    variable="Shortwave surface downward direct normal irradiance",
    resolution="3 km",
    timescale="hourly",
    data_type="Gridded",
    latitude=(stn_lat - 0.05, stn_lat + 0.05),
    longitude=(stn_lon - 0.06, stn_lon + 0.06),
    area_average="Yes",
    scenario=["Historical Climate", "SSP 3-7.0"],
    time_slice=(start_year, end_year + 1),
)

direct_irradiance_data = convert_to_local_time(
    direct_irradiance_data, stn_lon, stn_lat
)  # convert to local timezone, provide lon/lat because area average data lacks coordinates
direct_irradiance_data = direct_irradiance_data.sel(
    {"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")}
)
direct_irradiance_data = direct_irradiance_data.sel(simulation=data_models)

direct_irradiance_data.name = "Direct normal irradiance"  # rename for clarity
# total direct normal irradiance (accumulation of hourly data over a 24-hour period)
total_dni_data = direct_irradiance_data.resample(time="1D").sum()

Data converted to America/Los_Angeles timezone.


#### Step 1d: Load all variables
Now that we have all of our data retrieved and calculated, it is time to actually load the data into memory. Previously, xarray has lazily loaded the data, but we will actually grab it now. This will take approximately **7 minutes**. 

In [13]:
all_vars = xr.merge(
    [
        max_airtemp_data.squeeze(),
        min_airtemp_data.squeeze(),
        mean_airtemp_data.squeeze(),
        max_dewpt_data.squeeze(),
        min_dewpt_data.squeeze(),
        mean_dewpt_data.squeeze(),
        max_windspd_data.squeeze(),
        mean_windspd_data.squeeze(),
        total_ghi_data.squeeze(),
        total_dni_data.squeeze(),
    ]
)

In [ ]:
# load all indices in
all_vars = all_vars.compute()

In [ ]:
all_vars

In [ ]:
#! export so I don't have to compute all vars everytine - takes quite a while to compute usually.
# though sometimes as little as 12 minutes. Why? At present, I do not know.
# Save
all_vars.to_netcdf("all_vars.nc")

### Step 2: Calculate percentile-based hourly profile
Essentially, we are applying the standard profile methdology to a TMY.

In [ ]:
#! load saved all_vars eagerly
all_vars = xr.load_dataset("all_vars.nc")

In [ ]:
# retrieve data


##### Step 2a: Calculate percentile-based hourly profile

In [ ]:
def compute_profile(data: xr.DataArray, days_in_year: int = 365, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each simulation.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Check for simulation dimension
    has_simulation = "simulation" in data.dims
    if has_simulation:
        n_simulations = len(data.simulation)
        simulations = data.simulation.values
    else:
        n_simulations = 1
        simulations = [None]

    # Get all available time_delta data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

    # warming_levels = data.warming_level.values

    # Create helper function to extract meaningful simulation labels
    def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
        """Extract meaningful simulation label from simulation identifier."""
        if sim is None:
            return f"Sim_{sim_idx+1}"

        sim_str = str(sim)
        if "WRF_" in sim_str:
            # Parse simulation name format: WRF_GCM_params_scenario
            # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
            parts = sim_str.split("_")
            if len(parts) >= 4:
                gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
                params = parts[2]  # e.g., r11i1p1f1
                scenario = parts[3]  # e.g., historical+ssp245

                # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
                if "ssp" in scenario:
                    ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                    ssp = f"ssp{ssp_part}"
                else:
                    ssp = "hist"  # fallback for historical-only

                return f"{gcm}-{params}-{ssp}"
            elif len(parts) >= 2:
                # Fallback for shorter format
                return f"{parts[1]}-{sim_idx+1}"
            else:
                return f"Sim_{sim_idx+1}"
        else:
            # Ensure uniqueness by adding index for non-WRF format
            base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
            return f"{base_name}-{sim_idx+1}"

    # Process all data using quantile computation across years
    print(
        f"      ⚙️ Computing quantiles for {n_simulations} simulation(s)"
    )

    # Initialize storage for profiles
    profile_data = {}

    # Progress tracking
    # total_combinations = len(warming_levels) * n_simulations
    #! no warming levels
    total_combinations = n_simulations
    with tqdm(
        total=total_combinations,
        desc="      Computing profiles",
        unit="combo",
        leave=False,
    ) as pbar:

        #! no warming levels
        # for wl_idx, wl in enumerate(warming_levels):
        for sim_idx, sim in enumerate(simulations):
            # Get simulation label
            sim_label = _get_simulation_label(sim, sim_idx)

            # Select data for this warming level and simulation combination
            if has_simulation:
                # subset_data = data.isel(warming_level=wl_idx, simulation=sim_idx)
                subset_data = data.isel(simulation=sim_idx)
                # else:
                #     subset_data = data.isel(warming_level=wl_idx)
                print(f'imulatation {sim_idx} not present in dataset.')

            # Group by hour_of_year and find the actual data value closest to the quantile
            # This gives us the actual data point closest to the q-th quantile for each of the 8760 hours
            # Load data to avoid dask chunking issues with quantile
            if hasattr(subset_data.data, "chunks"):
                # If it's a dask array, load it into memory
                subset_data = subset_data.compute()

            def _closest_to_quantile(dat: xr.DataArray) -> xr.DataArray:
                """Find the actual data value closest to the specified quantile."""
                # Stack all dimensions except time_delta into a single dimension
                stacked = dat.stack(all_dims=list(dat.dims))
                # Compute the target quantile value
                target_quantile = stacked.quantile(q, dim="all_dims")
                # Find the index of the value closest to the quantile
                closest_idx = abs(stacked - target_quantile).argmin(dim="all_dims")
                # Return the actual data value at that index
                return xr.DataArray(stacked.isel(all_dims=closest_idx).values)

            profile_1d = subset_data.groupby("hour_of_year").map(
                _closest_to_quantile
            )

            # Reshape to (days_in_year, 24) for the final DataFrame
            profile_reshaped = profile_1d.values.reshape(
                days_in_year, hours_per_day
            )

            # Store the profile
            #! what is going on here?
            #key = (f"WL_{wl}", sim_label)
            key = sim_label
            profile_data[key] = profile_reshaped

            pbar.update(1)

    # # Create the multi-index DataFrame structure
    # df_profile = _construct_profile_dataframe(
    #     profile_data=profile_data,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )

    # # Determine which formatting function to use based on the structure
    # _format_based_on_structure(df_profile)

    # # Prepare metadata dictionary
    # metadata = {
    #     "quantile": q,
    #     "method": "8760 analysis - actual data closest to quantile across 30 years",
    #     "description": f"Climate profile computed using actual data values closest to the {q*100:.0f}th percentile of hourly data",
    # }

    # # Add original data attributes if available
    # if hasattr(data, "attrs"):
    #     if "units" in data.attrs:
    #         metadata["units"] = data.attrs["units"]
    #     if "extended_description" in data.attrs:
    #         metadata["extended_description"] = data.attrs["extended_description"]
    #     if "variable_id" in data.attrs:
    #         metadata["variable_name"] = data.attrs["variable_id"]
    #     elif hasattr(data, "name") and data.name:
    #         metadata["variable_name"] = data.name

    # # Set all metadata using the helper function
    # set_profile_metadata(df_profile, metadata)

    # print(f"      ✅ Profile computation complete! Final shape: {df_profile.shape}")
    # print(
    #     f"         With index: {df_profile.index.name}, columns: {df_profile.columns.names}"
    # )
    # if hasattr(data, "attrs") and "units" in data.attrs:
    #     print(f"         Units: {data.attrs['units']}")

    return profile_data

In [ ]:
test = compute_profile(all_vars, q=0.9)

In [ ]:
test

##### Step 2b: Drop volcano years (Pinatubo 1991-1994)

In [ ]:
# Remove the years for the Pinatubo eruption
test_updated = test.where(
    (~test.year.isin([1991, 1992, 1993, 1994])), np.nan, drop=True
)

### Step 3: Variable weighting according to NREL methods

##### Step 3a: Weight variables according to NREL method

In [ ]:
def compute_weighted_fs(da_fs):
    """Weights the F-S statistics based on TMY3 methodology"""
    weights_per_var = {
        "Daily max air temperature": 1 / 20,
        "Daily min air temperature": 1 / 20,
        "Daily mean air temperature": 2 / 20,
        "Daily max dewpoint temperature": 1 / 20,
        "Daily min dewpoint temperature": 1 / 20,
        "Daily mean dewpoint temperature": 2 / 20,
        "Daily max wind speed": 1 / 20,
        "Daily mean wind speed": 1 / 20,
        "Global horizontal irradiance": 5 / 20,
        "Direct normal irradiance": 5 / 20,
    }

    for var, weight in weights_per_var.items():
        # Multiply each variable by it's appropriate weight
        da_fs[var] = da_fs[var] * weight
    return da_fs

In [ ]:
weighted_fs = compute_weighted_fs(test_updated)

##### Step 3b: Select candidate months for consideration

In [ ]:
# Sum
weighted_fs_sum = (
    weighted_fs.to_array().sum(dim=["variable", "bin_number"]).drop(["data"])
)

# Pass the weighted F-S sum data for simplicity
ds = weighted_fs_sum

df_list = []
num_values = (
    1  # Selecting the top value for now, persistence statistics calls for top 5
)
for sim in ds.simulation.values:
    for mon in ds.month.values:
        da_i = ds.sel(month=mon, simulation=sim)
        top_xr = da_i.sortby(da_i, ascending=True)[:num_values].expand_dims(
            ["month", "simulation"]
        )
        top_df_i = top_xr.to_dataframe(name="top_values")
        df_list.append(top_df_i)

# Concatenate list together for all months and simulations
top_df = pd.concat(df_list).drop(columns=["top_values"]).reset_index()
top_df

### Step 4: Generate XMY data outputs 

Generally, the following data is outputted using the TMY months:
- Date & time (UTC)
- Air temperature at 2m [°C]
- Dew point temperature [°C]
- Relative humidity [%]
- Global horizontal irradiance [W/m2]
- Direct normal irradiance [W/m2]
- Diffuse horizontal irradiance [W/m2]
- Downwelling infrared radiation [W/m2]
- Wind speed at 10m [m/s]
- Wind direction at 10m [°]
- Surface air pressure [Pa]

The following function will retrieve all of this data for the designated TMY month and concatenate it together into a pandas dataframe so that we can export it as a csv file. We'll do this next. 

In [ ]:
def generate_tmy_data(top_df, type):
    """Generate typical meteorological year data
    Output will be a list of dataframes per simulation.
    Print statements throughout the function indicate to the user the progress of the computatioconvert_to_local_time   Parameters
    -----------
    top_df: pd.DataFrame
        Table with column values month, simulation, and year
        Each month-sim-yr combo represents the top candidate that has the lowest weighted sum from the FS statistic
    type: str, "tmy" or "xmy"
        Pass in which type of climate profile to generate

    Returns
    --------
    dict of str:pd.DataFrame
        Dictionary in the format of {simulation:TMY corresponding to that simulation}

    """
    ## ================== GET DATA FROM CATALOG ==================
    vars_and_units = {
        "Air Temperature at 2m": "degC",
        "Dew point temperature": "degC",
        "Relative humidity": "[0 to 100]",
        "Instantaneous downwelling shortwave flux at bottom": "W/m2",
        "Shortwave surface downward direct normal irradiance": "W/m2",
        "Shortwave surface downward diffuse irradiance": "W/m2",
        "Instantaneous downwelling longwave flux at bottom": "W/m2",
        "Wind speed at 10m": "m s-1",
        "Wind direction at 10m": "degrees",
        "Surface Pressure": "Pa",
    }

    # Loop through each variable and grab data from catalog
    all_vars_list = []
    print("STEP 1: RETRIEVING HOURLY DATA FROM CATALOG\n")
    for var, units in vars_and_units.items():
        print(f"Retrieving data for {var}", end="... ")
        data_by_var = get_data(
            variable=var,
            resolution="3 km",
            timescale="hourly",
            data_type="Gridded",
            units=units,
            latitude=(stn_lat - 0.05, stn_lat + 0.05),
            longitude=(stn_lon - 0.06, stn_lon + 0.06),
            area_average="No",
            scenario=["Historical Climate", "SSP 3-7.0"],
            time_slice=(start_year, end_year + 1),
        )
        data_by_var = convert_to_local_time(data_by_var)  # convert to local timezone.
        data_by_var = data_by_var.sel(
            {"time": slice(f"{start_year}-01-01", f"{end_year}-12-31")}
        )  # get desired time slice in local time
        data_by_var = get_closest_gridcell(
            data_by_var, stn_lat, stn_lon, print_coords=False
        )  # retrieve only closest gridcell
        data_by_var = data_by_var.sel(
            simulation=data_models
        )  # Subset for only the models that have solar variables

        # Drop unwanted coords
        data_by_var = data_by_var.squeeze().drop(
            ["lakemask", "landmask", "x", "y", "Lambert_Conformal"]
        )

        all_vars_list.append(data_by_var)  # Append to list
        print("complete!")

    # Merge data from all variables into a single xr.Dataset object
    all_vars_ds = xr.merge(all_vars_list)

    ## ================== CONSTRUCT TMY ==================
    print(
        "\nSTEP 2: CALCULATING TYPICAL METEOROLOGICAL YEAR PER MODEL SIMULATION\nProgress bar shows code looping through each month in the year.\n"
    )
    tmy_df_all = {}
    for sim in all_vars_ds.simulation.values:
        df_list = []
        print(f"Calculating TMY for simulation: {sim}")
        for mon in tqdm(np.arange(1, 13, 1)):
            # Get year corresponding to month and simulation combo
            year = top_df.loc[
                (top_df["month"] == mon) & (top_df["simulation"] == sim)
            ].year.item()

            # Select data for unique month, year, and simulation
            data_at_stn_mon_sim_yr = all_vars_ds.sel(
                simulation=sim, time=f"{mon}-{year}"
            ).expand_dims("simulation")

            # Reformat as dataframe
            df_by_mon_sim_yr = data_at_stn_mon_sim_yr.to_dataframe()
            df_by_mon_sim_yr = df_by_mon_sim_yr.reset_index()

            # Reformat time index to remove seconds
            df_by_mon_sim_yr["time"] = pd.to_datetime(
                df_by_mon_sim_yr["time"].values
            ).strftime("%Y-%m-%d %H:%M")
            df_list.append(df_by_mon_sim_yr)

        # Concatenate all DataFrames together
        tmy_df_by_sim = pd.concat(df_list)
        tmy_df_all[sim] = tmy_df_by_sim

    return tmy_df_all  # Return dict of TMY by simulation

In the next cell, we will run the `generate_tmy_data` function which will retrieve, subset, and format the data for each month according to the TMY months for all requested variables. We have included print statements so you can watch the progress for each variable in each month as it builds. 

<span style="color:#FF0000">**Note**: <span style="color:#000000"> This will take time! On the Analytics Engine JupyterHub, this takes approximately 22 minutes. Progress bars will indicate the status of generating the TMY data for each simulation. 

In [ ]:
tmy_data_to_export = generate_tmy_data(top_df)

Let's observe what the TMY data looks like for one of the simulations:

In [ ]:
simulation = "WRF_EC-Earth3_r1i1p1f1"
tmy_data_to_export[simulation].head(5)

Next, we visualize the TMY data itself.

In [ ]:
tmy_data_to_export[simulation].plot(
    x="time",
    y=[
        "Air Temperature at 2m",
        "Dew point temperature",
        "Relative humidity",
        "Instantaneous downwelling shortwave flux at bottom",
        "Shortwave surface downward direct normal irradiance",
        "Shortwave surface downward diffuse irradiance",
        "Instantaneous downwelling longwave flux at bottom",
        "Wind speed at 10m",
        "Wind direction at 10m",
        "Surface Pressure",
    ],
    title=f"Typical Meteorological Year ({simulation})",
    subplots=True,
    figsize=(10, 8),
    legend=True,
)

Lastly, let's export the TMY data below as csv files. There will be a file per simulation downloaded. When utilizing TMY data in your own workflows, we recommend that **all simulations are considered** in your analyses, especially for future scenarios. Note, if you are working with a custom location, please also provide the optional `stn_elev` argument to `write_tmy_file`; if no elevation value is provided, an elevation value of 0.0 is set as the default. 

In [ ]:
for sim, tmy in tmy_data_to_export.items():
    filename = "TMY_{0}_{1}".format(
        stn_name.replace(" ", "_").replace("(", "").replace(")", ""), sim
    ).lower()
    write_tmy_file(
        filename,
        tmy_data_to_export[sim],
        (start_year, end_year),
        stn_name,
        stn_code,
        stn_lat,
        stn_lon,
        stn_state,
        file_ext="epw",
    )

### Troubleshooting

In [ ]:
top_df

In [ ]:
data_by_var = get_data(
    variable="Air Temperature at 2m",
    resolution="3 km",
    timescale="hourly",
    data_type="Gridded",
    units="degC",
    latitude=(stn_lat - 0.05, stn_lat + 0.05),
    longitude=(stn_lon - 0.06, stn_lon + 0.06),
    area_average="No",
    scenario=["Historical Climate", "SSP 3-7.0"],
    time_slice=(start_year, end_year + 1),
)
data_by_var_test = convert_to_local_time(data_by_var)

In [ ]:
data_by_var